# 09 — Homologar con el paper (Zhao ICLR 2022 / PCT)

**Objetivo:** dejar a `mlp_v1_p1s1_ppo.pt` y a PCT en las **mismas
condiciones del paper** para poder decir **supera / empata / inferior**.

Zhao Table 1 compara DRL online así: misma tarea, **mismo bin cerrado**,
mismas secuencias, **Uti.** = volumen contenido / volumen del bin,
**Num.** = ítems completamente dentro. Una pila con hn > H no es
solución de ese problema.

Instancia aquí: O3DBP p=1 s=1, europalé **1200×800×2000 mm**, pedido
fijo `00100408` (el único con xkpi publicado de PCT). No se copian las
tablas 10×10×10. Setting alineado con el default de PCT (Setting 2:
geometría + hasta 6 giros; Kagerer usó parámetros default).

PCT: si el packing plan LFS está materializado, se recalculan Uti./Num.
con la misma contención. Si no, hn publicado 2.105 m ⇒ **no factible**
en el bin de 2 m.

Este notebook escribe el JSON y el lado Zhao. El `10` sella la frase.
No se usa ηutil como veredicto.

In [1]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
if HERE.name == "notebooks":
    ML_ROOT = HERE.parent
else:
    ML_ROOT = HERE if (HERE / "src_ml").is_dir() else Path("/home/edmundo/packing-services/online_policy_ml")
REPO_ROOT = ML_ROOT.parent
for p in (ML_ROOT / "src_ml", REPO_ROOT / "src"):
    s = str(p)
    if s not in sys.path:
        sys.path.insert(0, s)

from bootstrap import setup
setup()

from bedbpp_eval import (
    MARCO_ZHAO,
    kpis_kagerer,
    kpis_zhao,
    kpis_zhao_from_plan,
    load_pct_plan,
    packing_plan_actions,
    pct_zhao_publicado,
    veredicto_zhao,
)
from evaluate import write_report
from paths import DEMO_BED_BPP, PCT_PACKING_PLAN, REPORTS_DIR, model_path_mlp_ppo_production
from problems import order_to_problem
from splits import load_orders
from packing_services.algorithms.drl_policy_3d_bpp import DRLPolicy3DBPP

ORDER_IDS = ["00100408"]  # fijo *antes* de ver Uti.
ckpt = model_path_mlp_ppo_production(1, 1)
assert ckpt.is_file(), f"Falta el PPO del 05: {ckpt}"
orders = load_orders(DEMO_BED_BPP)
pct_plans = load_pct_plan(PCT_PACKING_PLAN)

plan = {}
rows = []
pct_rows = {}
for oid in ORDER_IDS:
    problem = order_to_problem(
        orders, oid, lookahead_p=1, select_s=1,
        algorithm_name="drl_policy_3d_bpp", model_path=str(ckpt),
    )
    c0 = problem.containers[0]
    assert (round(c0.length), round(c0.width), round(c0.height)) == (1200, 800, 2000), (
        f"{oid}: bin {c0.length, c0.width, c0.height} ≠ 1200×800×2000"
    )
    solution = DRLPolicy3DBPP().run(problem)
    zhao = kpis_zhao(problem, solution)
    kag = kpis_kagerer(problem, solution)
    flags = problem.constraints
    row = {
        **kag,
        **zhao,
        "order_id": oid,
        "method": "mlp_v1_p1s1_ppo",
        "target": (orders[oid].get("properties") or {}).get("target"),
        "allow_rotation": bool(flags.allow_rotation),
        "basic_stability": bool(flags.basic_stability),
        "max_weight": bool(flags.max_weight),
    }
    rows.append(row)
    plan[oid] = packing_plan_actions(problem, solution)

    if pct_plans is not None and oid in pct_plans:
        pct_rows[oid] = {
            **kpis_zhao_from_plan(pct_plans[oid], n_order=row["N"]),
            "source": str(PCT_PACKING_PLAN),
            "eta_util": None,
        }
    else:
        pct_rows[oid] = pct_zhao_publicado(
            n_order=row["N"], eta_util=0.614, hn_m=2.105, nu=0,
        )

    print(
        oid,
        "nosotros: Uti.", row["uti"], "Num.", row["num"],
        "factible", row["feasible"], "hn", row["hn_m"],
        "| PCT: Uti.", pct_rows[oid]["uti"], "Num.", pct_rows[oid]["num"],
        "factible", pct_rows[oid]["feasible"], "hn", pct_rows[oid]["hn_m"],
    )

assert all((r.get("target") or "").lower() == "euro-pallet" for r in rows)

veredictos = {
    r["order_id"]: veredicto_zhao(r, pct_rows[r["order_id"]]) for r in rows
}
for oid, v in veredictos.items():
    print(oid, v["frase"])

setting_nuestro = {
    "tarea": "O3DBP-1-1 (Zhao online; Kagerer corrió PCT en O3DBP)",
    "bin_mm": [1200, 800, 2000],
    "orientaciones": "hasta |O|=6 (Setting 2, default del código PCT)",
    "estabilidad": "basic_stability=False (Setting 2, no Setting 1)",
    "peso": "max_weight=True (extra vs Zhao Setting 2)",
    "alineacion": "Setting 2 + peso; bin cerrado 2000 mm",
}

write_report(
    REPORTS_DIR / "09_homologar_pct.json",
    {
        "marco_zhao": MARCO_ZHAO,
        "setting_nuestro": setting_nuestro,
        "order_ids": ORDER_IDS,
        "juez": (
            "Zhao Table 1: Uti./Num. solo ítems contenidos en 1200×800×2000 mm. "
            "PCT: packing plan si LFS materializado; si no, xkpi publicado "
            "(hn=2.105 m ⇒ no factible). ηutil Kagerer es contexto."
        ),
        "nuestro": rows,
        "pct": pct_rows,
        "veredicto_zhao": veredictos,
        "packing_plan_nuestro": plan,
        "cobertura_pct": (
            "plan LFS materializado" if pct_plans is not None
            else "puntero Git LFS; PCT por xkpi publicado de 00100408"
        ),
        "sigma_algo": "fuera de alcance (sin Blender); no es la columna Zhao",
    },
)
print("Siguiente: 10_comparar_pct.ipynb")

00100408 nosotros: Uti. 0.646376 Num. 26 factible True hn 1.97 | PCT: Uti. None Num. None factible False hn 2.105
00100408 En el pedido homologado, O3DBP p=1 s=1, bin 1200×800×2000 mm (Zhao ICLR 2022: Uti./Num., contención), `mlp_v1_p1s1_ppo.pt` supera PCT. nuestra solución es factible en el bin 1200×800×2000 mm; la de PCT publicada no (hn > 2 m). En el protocolo Zhao una colocación fuera del bin no es solución. No es ranking sobre BED-BPP ni Σalgo.
Siguiente: 10_comparar_pct.ipynb
